In [1]:
!pip install tensorflow scikit-learn pandas matplotlib --quiet
print("✅ Librerías instaladas.")

✅ Librerías instaladas.


In [2]:
import os, gc, time, warnings, random, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, Activation, MaxPooling2D,
    GlobalAveragePooling2D, Dense, Dropout, SpatialDropout2D
)
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

tf.keras.mixed_precision.set_global_policy('mixed_float16')

param_names = ["T", "Jex2", "Jex3", "Jex4", "Kan1", "KanS", "Hex", "KDM"]
param_labels = {
    "T": r"$T^{(0)}$", "Jex2": r"$\tilde{J}_2$", "Jex3": r"$\tilde{J}_3$",
    "Jex4": r"$\tilde{J}_4$", "Kan1": r"$\tilde{K}_{an1}$", "KanS": r"$\tilde{K}_{anS}$",
    "Hex": r"$\tilde{H}_{ex}$", "KDM": r"$\tilde{K}_{DM}$",
}
N_PARAMS = 8
T_IDX = 0
identificables = ["T", "Jex2", "Kan1", "KanS", "Hex", "KDM"]

print(f"TF: {tf.__version__} | GPUs: {len(gpus)}")

2026-07-05 21:33:59.767829: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-05 21:33:59.842567: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-05 21:34:01.434739: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TF: 2.20.0 | GPUs: 1


In [3]:
DATASET_PATH = "/workspace/Preprocess/latent_space_outputs/dataset_final_estructuras.npz"

data = np.load(DATASET_PATH, allow_pickle=True)
X = np.copy(data['img']).astype(np.float32)
y = np.copy(data['params']).astype(np.float32)

print(f"X: {X.shape} | y: {y.shape} | rango X: [{X.min():.3f}, {X.max():.3f}]")

X: (166141, 39, 39, 1) | y: (166141, 8) | rango X: [-1.000, 1.000]


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1765, random_state=SEED)

scaler = MinMaxScaler()
y_train_scaled = scaler.fit_transform(y_train)
y_val_scaled   = scaler.transform(y_val)
y_test_scaled  = scaler.transform(y_test)
data_range = scaler.data_max_ - scaler.data_min_

print(f"Train: {X_train.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}")

Train: 116,293 | Val: 24,926 | Test: 24,922


In [5]:
# ═══════════════════════════════════════════════════════════════
# ABLATION: pesos todos = 1  (SIN reponderación por orden)
# El modelo CON weights usaba order_based_weights_from_proxy aquí.
# Al ponerlos en 1, el único cambio vs el CNN-Aug-Weighted es el W.
# ═══════════════════════════════════════════════════════════════
sample_weights_train = np.ones(len(X_train), dtype=np.float32)

print(f"Weights UNIFORMES — todos = {sample_weights_train[0]:.1f} "
      f"(ablation: sin reponderación por orden)")

Weights UNIFORMES — todos = 1.0 (ablation: sin reponderación por orden)


In [6]:
BATCH_SIZE = 256

def make_ds(X_, y_, w_=None, shuffle=False):
    if w_ is None:
        w_ = np.ones(len(X_), dtype=np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X_, y_, w_))
    if shuffle:
        ds = ds.shuffle(10000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train_scaled, sample_weights_train, shuffle=True)
val_ds   = make_ds(X_val,   y_val_scaled)
test_ds  = make_ds(X_test,  y_test_scaled)

for xb, yb, wb in train_ds.take(1):
    print(f"x: {xb.shape}  y: {yb.shape}  w: {wb.shape}  (w medio: {wb.numpy().mean():.2f})")

I0000 00:00:1783287287.063965   72821 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 106730 MB memory:  -> device: 0, name: NVIDIA H200, pci bus id: 0000:5d:00.0, compute capability: 9.0


x: (256, 39, 39, 1)  y: (256, 8)  w: (256,)  (w medio: 1.00)


2026-07-05 21:34:50.173740: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [7]:
class PhysicalAugmentation(tf.keras.layers.Layer):
    """Rotaciones 90° + flips aleatorios a nivel de batch, en GPU.
    Solo activa en entrenamiento."""
    def call(self, x, training=None):
        if not training:
            return x
        k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
        x = tf.image.rot90(x, k=k)
        x = tf.cond(tf.random.uniform([]) > 0.5,
                    lambda: tf.image.flip_left_right(x), lambda: x)
        x = tf.cond(tf.random.uniform([]) > 0.5,
                    lambda: tf.image.flip_up_down(x), lambda: x)
        return x

    def get_config(self):
        return super().get_config()


def conv_block(x, filters, l2=1e-4, sdrop=0.1):
    x = Conv2D(filters, 3, padding='same',
               kernel_regularizer=regularizers.l2(l2))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, 3, padding='same',
               kernel_regularizer=regularizers.l2(l2))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = SpatialDropout2D(sdrop)(x)
    return x


def build_light_cnn(n_params=8):
    inputs = Input(shape=(39, 39, 1))
    x = PhysicalAugmentation()(inputs)        # augment EN GPU (solo training)

    x = conv_block(x, 32)                      # 39x39
    x = MaxPooling2D(2)(x)                     # 19x19
    x = conv_block(x, 64)
    x = MaxPooling2D(2)(x)                     # 9x9
    x = conv_block(x, 128)
    x = MaxPooling2D(2)(x)                     # 4x4
    x = conv_block(x, 256, sdrop=0.2)

    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu',
              kernel_regularizer=regularizers.l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(n_params, activation='linear', dtype='float32')(x)

    return Model(inputs, outputs, name="light_cnn_noweights")

model = build_light_cnn()
print(f"Parámetros totales: {model.count_params():,}")

Parámetros totales: 1,244,392


In [8]:
model.compile(optimizer=Adam(1e-3), loss='mse', metrics=['mae'])
print("✅ Compilado con MSE.")

✅ Compilado con MSE.


In [9]:
EPOCHS = 60

early_stop = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=5, min_lr=1e-7)

t0 = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    callbacks=[early_stop, reduce_lr], verbose=1)
total_time = time.time() - t0
print(f"\n⏱ {total_time/60:.1f} min | Épocas: {len(history.history['loss'])}")

Epoch 1/60


2026-07-05 21:35:23.014452: I external/local_xla/xla/service/service.cc:163] XLA service 0x776d34002b10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-07-05 21:35:23.014498: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA H200, Compute Capability 9.0
2026-07-05 21:35:23.173823: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:108] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. light_cnn_noweights_1/physical_augmentation_1/random_uniform
2026-07-05 21:35:23.174027: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:62] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. light_cnn_noweights_1/physical_augmentation_1/random_uniform_1/RandomUniform
2026-07-05 21:35:23.178084: W tensorflo

  7/455 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 2.5143 - mae: 1.1528     

2026-07-05 21:35:49.439055: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion_26', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1783287349.529103   73327 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


453/455 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.4024 - mae: 0.3319

2026-07-05 21:35:54.156449: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator light_cnn_noweights_1/physical_augmentation_1/rot90/Assert/AssertGuard/Assert
2026-07-05 21:35:55.990494: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-07-05 21:35:55.990603: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-07-05 21:35:55.990624: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the 

455/455 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 0.4017 - mae: 0.3314

2026-07-05 21:36:21.470638: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion_39', 8 bytes spill stores, 8 bytes spill loads

2026-07-05 21:36:24.843410: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-07-05 21:36:26.016059: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_418', 12 bytes spill stores, 12 bytes spill loads



455/455 ━━━━━━━━━━━━━━━━━━━━ 71s 85ms/step - loss: 0.4017 - mae: 0.3314 - val_loss: 0.1220 - val_mae: 0.0931 - learning_rate: 0.0010
Epoch 2/60
455/455 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.1161 - mae: 0.1066 - val_loss: 0.0965 - val_mae: 0.0713 - learning_rate: 0.0010
Epoch 3/60
455/455 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0873 - mae: 0.0784 - val_loss: 0.0786 - val_mae: 0.0724 - learning_rate: 0.0010
Epoch 4/60
455/455 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 0.0679 - mae: 0.0725 - val_loss: 0.0621 - val_mae: 0.0672 - learning_rate: 0.0010
Epoch 5/60
455/455 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 0.0521 - mae: 0.0697 - val_loss: 0.0475 - val_mae: 0.0708 - learning_rate: 0.0010
Epoch 6/60
455/455 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0397 - mae: 0.0676 - val_loss: 0.0398 - val_mae: 0.0742 - learning_rate: 0.0010
Epoch 7/60
455/455 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.0313 - mae: 0.0662 - val_loss: 0.0338 - val_mae: 0.0826 - learning_rate: 0.0010
Epoch 8/6

In [10]:
y_pred = scaler.inverse_transform(model.predict(test_ds, verbose=0))
y_true = y_test.copy()

rows = []
for i, name in enumerate(param_names):
    rows.append({"Parameter": name,
                 "MAE": round(mean_absolute_error(y_true[:, i], y_pred[:, i]), 4),
                 "R²": round(r2_score(y_true[:, i], y_pred[:, i]), 4)})
df_m = pd.DataFrame(rows).set_index("Parameter").sort_values("R²", ascending=False)
print(df_m.to_string())

r2_ident = df_m.loc[identificables, "R²"].mean()
print(f"\nR² promedio (identificables): {r2_ident:.4f}")
print(f"Referencia CNN-Aug-CON-W: (compara con el R² ident de tu modelo con weights)")

2026-07-05 21:41:34.939446: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


              MAE      R²
Parameter                
KDM        0.0179  0.9942
Hex        0.0124  0.9577
T          0.5306  0.9195
Kan1       0.0604  0.8289
Jex2       0.0222  0.7738
KanS       0.0275  0.6483
Jex3       0.0282  0.0044
Jex4       0.0219  0.0023

R² promedio (identificables): 0.8537
Referencia CNN-Aug-CON-W: (compara con el R² ident de tu modelo con weights)


In [11]:
SAVE_DIR = "/workspace/Models/Inverse/ModelWeights"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(f"{SAVE_DIR}/light_cnn_noweights_estructuras.keras")
np.savez("/workspace/Process/light_cnn_noweights_resultados.npz",
         y_true=y_true, y_pred=y_pred,
         loss=history.history['loss'], val_loss=history.history['val_loss'])
print(f"✅ Guardado: {SAVE_DIR}/light_cnn_noweights_estructuras.keras")

✅ Guardado: /workspace/Models/Inverse/ModelWeights/light_cnn_noweights_estructuras.keras


In [12]:
import numpy as np
import pandas as pd
from scipy import stats

data = np.load(DATASET_PATH, allow_pickle=True)
y = np.copy(data['params']).astype(np.float32)
param_names = ["T", "Jex2", "Jex3", "Jex4", "Kan1", "KanS", "Hex", "KDM"]

rows = []
for i, name in enumerate(param_names):
    v = y[:, i]
    rows.append({
        "Param": name,
        "min": round(v.min(), 3), "max": round(v.max(), 3),
        "mean": round(v.mean(), 3), "std": round(v.std(), 3),
        "skew": round(float(stats.skew(v)), 3),        # sesgo: |skew|>1 = muy sesgado
        "kurtosis": round(float(stats.kurtosis(v)), 3),
        "% en un valor": round(100 * np.max(np.bincount(
            np.digitize(v, np.linspace(v.min(), v.max(), 50)))) / len(v), 1),
    })
df_dist = pd.DataFrame(rows).set_index("Param")
print(df_dist.to_string())
print("\nGuía: |skew| > 1 → distribución sesgada, MinMax le hace daño.")
print("      kurtosis alta → colas pesadas, StandardScaler ayuda.")

         min     max   mean    std   skew  kurtosis  % en un valor
Param                                                             
T      0.000  20.000  4.185  3.138  1.054     1.202            6.6
Jex2  -0.331   0.659  0.046  0.140  2.744     7.049           81.7
Jex3  -0.290   0.290 -0.001  0.073  0.101     6.342           81.6
Jex4  -0.234   0.235 -0.001  0.058 -0.091     6.674           81.8
Kan1   0.000   4.568  0.140  0.348  8.181    80.792           54.0
KanS   0.000   0.200  0.078  0.062  0.316    -1.153           17.5
Hex    0.000   1.198  0.044  0.100  4.783    25.817           54.8
KDM    0.000   1.200  0.621  0.382 -0.363    -1.055           18.7

Guía: |skew| > 1 → distribución sesgada, MinMax le hace daño.
      kurtosis alta → colas pesadas, StandardScaler ayuda.
